# Word-Level Dialect Analysis
Cross-compares every source word against every word in the DAT and DIT transcripts using:
- **Levenshtein similarity** (normalised by the longest word in the dataset)
- **Position score** (normalised gap between source and target word index)
- **Combined weighted score** (α · lev_sim + (1-α) · pos_sim)
- **Combined harmonic score** (F1-style, penalises low scores in either dimension)

In [14]:
import sys
import importlib
import pandas as pd
from pathlib import Path

# Make scripts/ importable when running from inside scripts/
sys.path.insert(0, str(Path("__file__").resolve().parent))

import utils
importlib.reload(utils)

from utils import (
    clean,
    build_word_comparison_df,
)

In [15]:
PROJECT_ROOT = Path("__file__").resolve().parent.parent
ANALYSIS_DIR = PROJECT_ROOT / "02-analysis-position"
DAT_TSV = ANALYSIS_DIR / "dialect-aware-transcript.tsv"
DIT_TSV = ANALYSIS_DIR / "dialect-ignorant-transcript.tsv"

## Load & merge data

In [16]:
df_dit = pd.read_csv(DIT_TSV, sep='\t', encoding='utf-8-sig')
df_dat = pd.read_csv(DAT_TSV, sep='\t', encoding='utf-8-sig')
df = pd.merge(df_dit, df_dat[['path', 'DAT']], on='path', how='inner')

print(f"Clips: {len(df)}")
df.head(3)

Clips: 100


,path,duration,sentence,sentence_source,client_id,dialect_region,canton,zipcode,age,gender,DIT,DAT
0,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,4.778662,Insgesamt habe ich einige Hunderttausend Frank...,news,300bb931-79ae-40ec-b989-3efd5e83f4c2,Zürich,ZH,8408.0,fourties,male,Insgesamt habe ich einige hundert Dusik vor un...,Insgesamt habe ich einige 100.000 Franken verl...
1,clips\6e084270-8d26-43d9-ba69-5e8ee793ab8c\dca...,5.294150,Welche Rolle hatte Rosenberg während des Holoc...,news,6e084270-8d26-43d9-ba69-5e8ee793ab8c,Zürich,ZG,6340.0,twenties,female,"Weli Rolle, höchter Uz stoodhergewerden von Ho...",Wer ironne hättet viele bands auf den einzugem...
2,clips\6858a37b-edd0-4fdf-871c-96d3b1bd3e21\82f...,5.802676,Das ist angesichts aller schlechten Optionen d...,news,6858a37b-edd0-4fdf-871c-96d3b1bd3e21,Zürich,ZH,8704.0,fourties,male,Das ist angesichts vor allen Schlachten Option...,Das ist angesichts von allen schlechten Option...


## Determine global normalisation constant

In [17]:
all_words = (
    df['sentence'].dropna().str.split().explode().tolist()
    + df['DIT'].dropna().str.split().explode().tolist()
    + df['DAT'].dropna().str.split().explode().tolist()
)
max_word_len = max(len(clean(w)) for w in all_words)
print(f"Longest word length (Levenshtein normaliser): {max_word_len}")

Longest word length (Levenshtein normaliser): 27


## Cross-comparison: every source word × every target position

In [18]:
# Global normalisation: divide by longest word across the entire dataset
word_df = build_word_comparison_df(df, max_word_len=max_word_len)
print(f"Total rows: {len(word_df):,}")
word_df.head(10)

Total rows: 9,441


,clip_id,src_word_index,target_word_index,is_same_position,position_score,src_word,dit_word,dat_word,dit_lev_sim,dat_lev_sim,dit_combined_weighted,dit_combined_harmonic,dat_combined_weighted,dat_combined_harmonic,dat_advantage,src_len,dit_len,dat_len
0,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,0,1,1.000,insgesamt,insgesamt,insgesamt,1.000,1.000,1.000,1.000,1.000,1.000,0.000,7,9,7
1,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,1,0,0.875,insgesamt,habe,habe,0.704,0.704,0.755,0.780,0.755,0.780,0.000,7,9,7
2,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,2,0,0.750,insgesamt,ich,ich,0.704,0.704,0.718,0.726,0.718,0.726,0.000,7,9,7
3,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,3,0,0.625,insgesamt,einige,einige,0.778,0.778,0.732,0.693,0.732,0.693,0.000,7,9,7
4,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,4,0,0.500,insgesamt,hundert,100000,0.741,0.667,0.669,0.597,0.617,0.572,-0.074,7,9,7
5,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,5,0,0.375,insgesamt,dusik,franken,0.704,0.667,0.605,0.489,0.579,0.480,-0.037,7,9,7
6,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,6,0,0.250,insgesamt,vor,verloren,0.667,0.667,0.542,0.364,0.542,0.364,0.000,7,9,7
7,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,7,0,0.125,insgesamt,und,None,0.704,0.667,0.530,0.212,0.504,0.211,-0.037,7,9,7
8,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,8,0,0.000,insgesamt,verloren,None,0.667,0.667,0.467,0.000,0.467,0.000,0.000,7,9,7
9,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,1,0,0,0.875,habe,insgesamt,insgesamt,0.704,0.704,0.755,0.780,0.755,0.780,0.000,7,9,7


## Summary statistics

In [19]:
word_df[['dit_lev_sim', 'dat_lev_sim', 'position_score',
         'dit_combined_weighted', 'dat_combined_weighted',
         'dit_combined_harmonic', 'dat_combined_harmonic'
         ]].describe().round(3)

,dit_lev_sim,dat_lev_sim,position_score,dit_combined_weighted,dat_combined_weighted,dit_combined_harmonic,dat_combined_harmonic
count,9441.000,9441.000,9441.000,9441.000,9441.000,9441.000,9441.000
mean,0.785,0.787,0.641,0.741,0.743,0.669,0.670
std,0.116,0.120,0.257,0.115,0.118,0.201,0.203
min,0.074,0.185,0.000,0.152,0.207,0.000,0.000
25%,0.741,0.741,0.455,0.665,0.669,0.558,0.559
50%,0.815,0.815,0.667,0.749,0.750,0.707,0.709
75%,0.852,0.889,0.857,0.822,0.824,0.816,0.820
max,1.000,1.000,1.000,1.000,1.000,1.000,1.000


---

## Pairwise Levenshtein-Normalisation (NED)

Same Cross-Comparison as above, but normalisation of Levenshtein distance happens per word pair:

$$\text{sim\_pw}(a, b) = 1 - \frac{\text{lev}(a, b)}{\max(\text{len}(a),\ \text{len}(b))}$$

In [20]:
# Pairwise normalisation: max(len(src), len(target)) per word pair — standard NED
word_df_pw = build_word_comparison_df(df, max_word_len=None)
print(f"Total rows: {len(word_df_pw):,}")
word_df_pw.head(20)

Total rows: 9,441


,clip_id,src_word_index,target_word_index,is_same_position,position_score,src_word,dit_word,dat_word,dit_lev_sim,dat_lev_sim,dit_combined_weighted,dit_combined_harmonic,dat_combined_weighted,dat_combined_harmonic,dat_advantage,src_len,dit_len,dat_len
0,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,0,1,1.000,insgesamt,insgesamt,insgesamt,1.000,1.000,1.000,1.000,1.000,1.000,0.000,7,9,7
1,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,1,0,0.875,insgesamt,habe,habe,0.111,0.111,0.340,0.197,0.340,0.197,0.000,7,9,7
2,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,2,0,0.750,insgesamt,ich,ich,0.111,0.111,0.303,0.193,0.303,0.193,0.000,7,9,7
3,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,3,0,0.625,insgesamt,einige,einige,0.333,0.333,0.421,0.434,0.421,0.434,0.000,7,9,7
4,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,4,0,0.500,insgesamt,hundert,100000,0.222,0.000,0.305,0.307,0.150,0.000,-0.222,7,9,7
5,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,5,0,0.375,insgesamt,dusik,franken,0.111,0.000,0.190,0.171,0.113,0.000,-0.111,7,9,7
6,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,6,0,0.250,insgesamt,vor,verloren,0.000,0.000,0.075,0.000,0.075,0.000,0.000,7,9,7
7,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,7,0,0.125,insgesamt,und,None,0.111,0.000,0.115,0.118,0.038,0.000,-0.111,7,9,7
8,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,8,0,0.000,insgesamt,verloren,None,0.000,0.000,0.000,0.000,0.000,0.000,0.000,7,9,7
9,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,1,0,0,0.875,habe,insgesamt,insgesamt,0.111,0.111,0.340,0.197,0.340,0.197,0.000,7,9,7


In [21]:
word_df[['dit_lev_sim', 'dat_lev_sim', 'position_score',
         'dit_combined_weighted', 'dat_combined_weighted',
         'dit_combined_harmonic', 'dat_combined_harmonic'
         ]].describe().round(3)

,dit_lev_sim,dat_lev_sim,position_score,dit_combined_weighted,dat_combined_weighted,dit_combined_harmonic,dat_combined_harmonic
count,9441.000,9441.000,9441.000,9441.000,9441.000,9441.000,9441.000
mean,0.785,0.787,0.641,0.741,0.743,0.669,0.670
std,0.116,0.120,0.257,0.115,0.118,0.201,0.203
min,0.074,0.185,0.000,0.152,0.207,0.000,0.000
25%,0.741,0.741,0.455,0.665,0.669,0.558,0.559
50%,0.815,0.815,0.667,0.749,0.750,0.707,0.709
75%,0.852,0.889,0.857,0.822,0.824,0.816,0.820
max,1.000,1.000,1.000,1.000,1.000,1.000,1.000
